# Cross-industry comparison: inspectable supplied inputs

Run this notebook in an existing Alpha Python environment with the repository's locked dependencies installed. These cells are tested locally with networking blocked. An actual hosted Colab deployment is not qualified by those tests.

The bundled example is synthetic. Replace its assessments with your own evidence, inspect trade-offs, and export the exact input and recomputed report. No provider, wallet, orchestrator, cloud tunnel, training process or external evidence URL is contacted. No API key or GPU is needed.

[Project notice](../../../DISCLAIMER_SNIPPET.md) · [Input schema and limitations](README.md) · [Original conceptual framework and flowcharts](CONCEPTUAL_FRAMEWORK.md)


## 1. Load the installed comparator

Use the same Python environment as the repository CLI. If the import fails, select the environment where Alpha and its hash-locked requirements are already installed. This notebook does not install or download software automatically.


In [ ]:
import copy
import hashlib
import json
from pathlib import Path
from alpha_factory_v1.demos.cross_industry_alpha_factory.opportunity_comparison import (
    EXAMPLE, MAX_BYTES, compare_opportunities, load_input,
)
print("Comparator ready. No provider or network request was made.")


## 2. Choose exact supplied input

Leave INPUT_PATH as None to use the clearly labeled synthetic example. Otherwise set it to a local JSON file following the documented schema. The loader rejects duplicate JSON keys, malformed UTF-8 and files over 1 MiB.


In [ ]:
INPUT_PATH = None  # For example: Path("my-opportunities.json")
source_mode = "synthetic-example" if INPUT_PATH is None else "supplied-input"
assessment = copy.deepcopy(load_input(EXAMPLE if INPUT_PATH is None else Path(INPUT_PATH)))
print(source_mode)
print(json.dumps(assessment, ensure_ascii=False, indent=2))


## 3. Edit and compare

Edit the explicit weights, thresholds, ratings and evidence below, then rerun this and subsequent cells. Ratings are supplied judgments, not measured returns or verified facts. Every run recomputes the report from the current input; no previous result is reused.


In [ ]:
# Example edit: assessment["constraints"]["max_risk"] = 40
report = compare_opportunities(assessment)
report["mode"] = source_mode
print(json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False))


## 4. Export without overwriting

The evidence includes the full supplied input and deterministic report. The report input digest identifies normalized input; the printed file digest identifies exact exported bytes. This is unsigned comparison evidence, not execution permission or financial approval. Choose a new filename if it already exists.


In [ ]:
current_report = compare_opportunities(assessment)
current_report["mode"] = source_mode
if report != current_report:
    raise ValueError("Input or report changed: rerun and inspect the comparison before export")
evidence_path = Path("cross-industry-evidence.json")
evidence = {"schema": "cross-industry-notebook-evidence-v1", "input": assessment, "report": report}
evidence_bytes = (json.dumps(evidence, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False) + "\n").encode("utf-8")
if len(evidence_bytes) > 4 * 1024 * 1024:
    raise ValueError("Evidence exceeds the 4 MiB notebook bound")
with evidence_path.open("xb") as destination:
    destination.write(evidence_bytes)
print(str(evidence_path), "sha256:" + hashlib.sha256(evidence_bytes).hexdigest())


## 5. Recover input and independently recompute

This cell ignores any saved report. It parses bounded strict JSON, validates the recovered input and recomputes the comparison. A stored result can never grant authority or replace validation.


In [ ]:
def unique_object(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError("Duplicate evidence key: " + key)
        result[key] = value
    return result

def reject_constant(value):
    raise ValueError("Non-finite JSON value: " + value)

with evidence_path.open("rb") as source:
    recovered_bytes = source.read(4 * 1024 * 1024 + 1)
if len(recovered_bytes) > 4 * 1024 * 1024:
    raise ValueError("Evidence exceeds the 4 MiB notebook bound")
recovered = json.loads(recovered_bytes.decode("utf-8"), object_pairs_hook=unique_object, parse_constant=reject_constant)
if not isinstance(recovered, dict) or set(recovered) != {"schema", "input", "report"} or recovered["schema"] != "cross-industry-notebook-evidence-v1":
    raise ValueError("Unexpected notebook evidence schema")
recomputed = compare_opportunities(recovered["input"])
print("Recomputed supplied input; stored report ignored.")
print(json.dumps(recomputed, ensure_ascii=False, indent=2, allow_nan=False))


## Historical deployment context and limits

The previous notebook attempted background orchestration, mock domain endpoints, public tunnels, hot-reloaded model uploads and continuous training. That walkthrough contained invalid Python and did not establish authenticated deployment, isolated training, meaningful domain outputs or production safety. It is no longer the runnable quick-start.

The original research diagrams and design discussion remain in README.md and CONCEPTUAL_FRAMEWORK.md. The separate legacy deployment shell script is advanced, unqualified material: do not run it as evidence that the supplied-input comparison needs services, public ports, credentials or model training. A future operational deployment needs its own reviewed configuration, authentication, isolation, resource limits and acceptance evidence.

This notebook's demonstrated outcome is bounded comparison and recoverable unsigned evidence. Human validation, provider execution and economic outcomes require separate work.


## Reopen the exact input in the original browser
The browser accepts bounded notebook evidence. For larger evidence envelopes, reopen this raw input file instead. Existing files are never overwritten.


In [ ]:
raw_input_path = Path("cross-industry-input.json")
raw_input_bytes = (json.dumps(assessment, ensure_ascii=False, separators=(",", ":"), allow_nan=False) + "\n").encode("utf-8")
if len(raw_input_bytes) > MAX_BYTES:
    raise ValueError("Raw input exceeds the 1 MiB bound")
with raw_input_path.open("xb") as target:
    target.write(raw_input_bytes)
print(raw_input_path, "sha256:" + hashlib.sha256(raw_input_bytes).hexdigest())
